# PubMed Retrieval with ModernPubMedBERT + FAISS

This notebook builds a FAISS index over `text_to_embed` using the Hugging Face model `lokeshch19/ModernPubMedBERT` and provides a simple retrieval function.

Notes:
- The model has a max sequence length of 2048 tokens. We enforce truncation to this limit.
- Embeddings use mean pooling over the last hidden state with attention mask.

In [15]:
import json
from pathlib import Path
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
import faiss

In [16]:
DATA_PATH = Path("/Users/ftzavellos/Law_and_Tech/drug_explanations/Code/preprocessing_output/semaglutide_pubmed.jsonl")
OUT_DIR = Path("/Users/ftzavellos/Law_and_Tech/drug_explanations/Code/retrieval_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "lokeshch19/ModernPubMedBERT"
MAX_LEN = 2048
BATCH_SIZE = 8

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [17]:
def load_records(path: Path):
    records = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records

records = load_records(DATA_PATH)
len(records)

1572

In [18]:
def get_text(rec: dict) -> str:
    return (rec.get("text_to_embed") or "").strip()

texts = [get_text(r) for r in records]
keep = [i for i,t in enumerate(texts) if t]
records = [records[i] for i in keep]
texts = [texts[i] for i in keep]
len(texts)

1572

In [19]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.to(device)
model.eval()

def count_tokens(texts):
    return [len(tokenizer.encode(t, add_special_tokens=True)) for t in texts]

lengths = count_tokens(texts)
over = sum(1 for n in lengths if n > MAX_LEN)
max_len = max(lengths) if lengths else 0
over, max_len

Token indices sequence length is longer than the specified maximum sequence length for this model (2402 > 2048). Running this sequence through the model will result in indexing errors


(1, 2402)

In [20]:
@torch.no_grad()
def embed_texts(batch_texts):
    enc = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="pt",
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    out = model(**enc)
    last = out.last_hidden_state
    mask = enc["attention_mask"].unsqueeze(-1)
    summed = (last * mask).sum(dim=1)
    counts = mask.sum(dim=1).clamp(min=1)
    mean = summed / counts
    return mean.cpu().numpy()

embs = []
for i in range(0, len(texts), BATCH_SIZE):
    batch = texts[i:i+BATCH_SIZE]
    embs.append(embed_texts(batch))

embs = np.vstack(embs).astype("float32")
embs.shape

(1572, 768)

In [26]:
dim = embs.shape[1]
index = faiss.IndexFlatIP(dim)

# Normalize for cosine similarity with IP
faiss.normalize_L2(embs)
index.add(embs)

index.ntotal

1572

In [27]:
# Save index + metadata
index_path = OUT_DIR / "semaglutide_pubmed_faiss.index"
meta_path = OUT_DIR / "semaglutide_pubmed_meta.jsonl"

faiss.write_index(index, str(index_path))

with meta_path.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=True) + "\n")

index_path, meta_path

(PosixPath('/Users/ftzavellos/Law_and_Tech/drug_explanations/Code/retrieval_outputs/semaglutide_pubmed_faiss.index'),
 PosixPath('/Users/ftzavellos/Law_and_Tech/drug_explanations/Code/retrieval_outputs/semaglutide_pubmed_meta.jsonl'))

In [28]:
def retrieve(query: str, k: int = 5):
    q_emb = embed_texts([query]).astype("float32")
    faiss.normalize_L2(q_emb)
    scores, idxs = index.search(q_emb, k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        rec = records[int(idx)]
        results.append({
            "score": float(score),
            "pmid": rec.get("pmid"),
            "doi": rec.get("doi"),
            "title": rec.get("title"),
            "journal": rec.get("journal"),
            "year": rec.get("year"),
            "abstract": rec.get("abstract"),
            "text_to_embed": rec.get("text_to_embed"),
        })
    return results


In [24]:
#retrieve("Semaglutide induces significant weight loss in adults with obesity.", k=15)

In [29]:
HARDCODED_QUERY = "Semaglutide induces significant weight loss in adults with obesity."

results_path = OUT_DIR / "retrieval_results.jsonl"
with results_path.open("w", encoding="utf-8") as f:
    hits = retrieve(HARDCODED_QUERY, k=15)
    f.write(json.dumps({
        "query": HARDCODED_QUERY,
        "articles": hits,
    }, ensure_ascii=False) + "\n")

results_path


PosixPath('/Users/ftzavellos/Law_and_Tech/drug_explanations/Code/retrieval_outputs/retrieval_results.jsonl')